# EDA — Predicción de Popularidad de Canciones en Spotify
### Evaluación Parcial N°2 — MLY1101 Machine Learning

Este notebook continúa el proyecto a partir del dataset limpio (`data/processed/spotify_tracks_clean.csv`), generado en `01_preparacion_datos.ipynb`. En ese dataset **cada fila es una canción única** (`track_id`) y los géneros están codificados como columnas `genre_*` con 0/1 (multi-hot encoding). Se realiza el análisis exploratorio de datos (EDA) y la evaluación de sesgos, ética y privacidad.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

df = pd.read_csv('../data/processed/spotify_tracks_clean.csv')
genre_cols = [c for c in df.columns if c.startswith('genre_')]

print(df.shape)
print('Canciones únicas:', df['track_id'].nunique(), '| columnas de género:', len(genre_cols))
df.head()

## 3. Análisis exploratorio de los datos (EDA)

### 3.1 Distribución de la variable objetivo: `popularity`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['popularity'], bins=30, ax=axes[0])
axes[0].set_title('Distribución de popularity')

sns.boxplot(x=df['popularity'], ax=axes[1])
axes[1].set_title('Boxplot de popularity')

plt.tight_layout()
plt.savefig('../images/distribucion_popularity.png', dpi=100)
plt.show()

print('Popularity = 0: {} canciones ({:.1%})'.format((df['popularity'] == 0).sum(), (df['popularity'] == 0).mean()))
print('Popularity > 70 ("top"): {} canciones ({:.1%})'.format((df['popularity'] > 70).sum(), (df['popularity'] > 70).mean()))
df['popularity'].describe()

**Hallazgo:** la distribución de `popularity` está sesgada hacia valores bajos-medios (media ≈ 33, mediana 33), con una acumulación notoria en 0 (**10,5%** de las canciones, 9.413). Esto sugiere que muchas canciones del dataset prácticamente no tuvieron reproducciones relevantes, lo cual es esperable en un catálogo tan amplio (89.740 canciones únicas de 114 géneros). Las canciones "top" (popularity > 70) son solo el **3,0%** (2.729): es una clase muy escasa, relevante para el KPI de canciones top.

### 3.2 Relación entre audio features y popularity

In [ ]:
num_cols = ['popularity','duration_ms','danceability','energy','key','loudness','mode',
            'speechiness','acousticness','instrumentalness','liveness','valence','tempo','time_signature']

corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación - Audio features vs Popularity')
plt.tight_layout()
plt.savefig('../images/correlacion_features.png', dpi=100)
plt.show()

corr['popularity'].sort_values(ascending=False)

**Hallazgo clave:** ninguna audio feature individual tiene una correlación fuerte con `popularity` (todas por debajo de 0,13 en valor absoluto). `loudness` (0,07) y `danceability` (0,06) son las más asociadas positivamente, e `instrumentalness` (-0,13) y `speechiness` (-0,05) las más asociadas negativamente. Esto indica que la popularidad de una canción **no se explica solo por sus características acústicas**: probablemente influyen más factores externos como el artista, la promoción, el género o la fecha de lanzamiento (no disponible en este dataset). Como las relaciones pueden no ser lineales, se probarán modelos más flexibles que la regresión lineal.

### 3.3 Popularidad por género musical

Con el multi-hot encoding una canción puede pertenecer a varios géneros; la popularidad promedio de un género se calcula sobre todas las canciones que tienen un 1 en su columna.

In [ ]:
pop_por_genero = pd.Series({
    c.replace('genre_', ''): df.loc[df[c] == 1, 'popularity'].mean() for c in genre_cols
}).sort_values(ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(10, 10))

pop_por_genero.head(15).plot(kind='barh', ax=axes[0], color='seagreen')
axes[0].invert_yaxis()
axes[0].set_title('Top 15 géneros con mayor popularidad promedio')
axes[0].set_xlabel('Popularity promedio')

pop_por_genero.tail(15).plot(kind='barh', ax=axes[1], color='indianred')
axes[1].invert_yaxis()
axes[1].set_title('15 géneros con menor popularidad promedio')
axes[1].set_xlabel('Popularity promedio')

plt.tight_layout()
plt.savefig('../images/popularidad_por_genero.png', dpi=100)
plt.show()

print('Mayor:', pop_por_genero.head(3).round(1).to_dict())
print('Menor:', pop_por_genero.tail(3).round(1).to_dict())

**Hallazgo:** a diferencia de las audio features, **el género sí marca una diferencia grande** en la popularidad promedio: `pop-film` (59), `k-pop` (57) y `chill` (54) lideran, mientras que `latin` (8), `romance` (3,5) e `iranian` (2,2) quedan en el extremo inferior. Esto sugiere que los géneros (`genre_*`) serán predictores relevantes para el modelo, más que las audio features individuales.

### 3.4 Canciones explícitas vs no explícitas

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x='explicit', y='popularity', data=df, ax=ax)
ax.set_title('Popularity según contenido explícito')
plt.tight_layout()
plt.savefig('../images/popularity_explicit.png', dpi=100)
plt.show()

df.groupby('explicit')['popularity'].agg(['mean', 'count'])

**Hallazgo:** las canciones explícitas tienen una popularidad promedio algo mayor (≈ 36,9 vs 32,9), pero son solo ~8,6% del catálogo (7.704 canciones).

## 4. Evaluación de sesgos, ética y privacidad de los datos

**Privacidad:** el dataset no contiene datos personales de usuarios (no hay IDs de oyentes, ubicación ni historial de reproducción individual); solo metadatos públicos de canciones (nombre, artista, álbum) y métricas agregadas calculadas por Spotify. El riesgo de privacidad es bajo.

**Sesgos identificados:**
- **Sesgo de género musical:** el dataset no tiene una distribución equilibrada de popularidad entre géneros: géneros mainstream (pop-film, k-pop) tienen popularidad promedio muy superior a géneros de nicho (iranian, romance). Un modelo entrenado con estos datos podría aprender a predecir sistemáticamente baja popularidad para géneros minoritarios, incluso si una canción específica es de buena calidad.
- **Sesgo de "estrella" (long tail):** ~10,5% de las canciones tienen popularity = 0 y solo ~3% superan 70, lo que refleja cómo unos pocos artistas/canciones concentran la mayoría de las reproducciones (fenómeno long-tail típico de plataformas de streaming). Esto puede sesgar el modelo hacia predecir valores medios-bajos.
- **Ausencia de variables temporales y de contexto:** no hay fecha de lanzamiento, país de origen del oyente, ni datos de campañas de marketing, factores que en la realidad influyen fuertemente en la popularidad. El modelo solo puede capturar la relación entre audio features/género y popularidad, no factores externos (esto es una limitación, no solo un sesgo).

**Mitigación propuesta:** documentar estas limitaciones en las conclusiones del proyecto y evitar presentar el modelo como una herramienta de decisión única para promocionar música; debe usarse como apoyo, no reemplazo del criterio humano/experto en A&R (Artists & Repertoire).